In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False


class PolynomialClassificationSuite:
    """
    Unified framework for polynomial root classification.
    """
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.rng = np.random.RandomState(random_state)

    
    # Quadratic, cubic, and quartic demonstrations
    def demonstrate_quadratic(self, n_samples=45000, verbose=True):
        """
        Demonstrate perfect quadratic classification using b²/ac discriminant.
        """
        if verbose:
            print("\n" + "="*80))
            print("Quadratic demonstration")
            print("="*80)
        
        # Generate base data
        n_base = 40000
        a = self.rng.uniform(-10, 10, n_base)
        b = self.rng.uniform(-10, 10, n_base)
        c = self.rng.uniform(-10, 10, n_base)
        
        # Compute discriminant feature
        ac = a * c
        epsilon = 1e-8
        b2_ac_base = np.where(np.abs(ac) > epsilon, b**2 / ac, 0)
        discriminant = b**2 - 4*a*c
        y_base = (discriminant < 0).astype(int)
        
        # Oversample near boundary (b²/ac ≈ 4) for better learning
        n_border = 5000
        b2_ac_border = self.rng.normal(loc=4.0, scale=0.2, size=n_border)
        b2_ac_border = np.clip(b2_ac_border, 0, 10)
        y_border = (b2_ac_border < 4).astype(int)
        
        # Combine and shuffle
        X = np.concatenate([b2_ac_base, b2_ac_border]).reshape(-1, 1)
        y = np.concatenate([y_base, y_border])
        
        # Shuffle using our RNG
        indices = self.rng.permutation(len(X))
        X, y = X[indices], y[indices]
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=self.random_state
        )
        
        # Train shallow tree
        clf = DecisionTreeClassifier(max_depth=2, random_state=self.random_state)
        clf.fit(X_train, y_train)
        
        # Evaluate
        y_pred = clf.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        bal_accuracy = balanced_accuracy_score(y_test, y_pred)
        
        if verbose:
            print(f"Accuracy: {accuracy:.5f}")
            print(f"Balanced Accuracy: {bal_accuracy:.5f}")
            print("\nLearned Rule:")
            print(export_text(clf, feature_names=['b²/ac'], max_depth=2))
        
        return {
            'accuracy': accuracy,
            'balanced_accuracy': bal_accuracy,
            'model': clf,
            'name': 'Quadratic'
        }
    
    def demonstrate_cubic(self, n_samples=10000, verbose=True):
        """
        Demonstrate cubic classification using β²/α³ ratio.
        """
        if verbose:
            print("\n" + "="*80)
            print("Cubic demonstration")
            print("="*80)
        
        # Generate data
        A = self.rng.uniform(-10, 10, n_samples)
        B = self.rng.uniform(-10, 10, n_samples)
        C = self.rng.uniform(-10, 10, n_samples)
        
        # Compute reduced form parameters
        a = A / 3.0
        b = B / 3.0
        alpha = a**2 - b
        beta = 2*a**3 - 3*a*b + C
        
        # Alpha_Beta = β²/α³
        alpha_beta = np.where(alpha > 0, (beta**2) / (alpha**3), np.inf)
        alpha_beta_finite = np.where(np.isinf(alpha_beta), 1e12, alpha_beta)
        
        # Ground truth via root computation
        def classify_cubic(A, B, C):
            roots = np.roots([1.0, A, B, C])
            return 0 if np.all(np.abs(roots.imag) < 1e-10) else 1
        
        y = np.array([classify_cubic(A[i], B[i], C[i]) for i in range(n_samples)])
        
        # Train single-split stump
        X = alpha_beta_finite.reshape(-1, 1)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=self.random_state, stratify=y
        )
        
        stump = DecisionTreeClassifier(max_depth=1, random_state=self.random_state)
        stump.fit(X_train, y_train)
        y_pred = stump.predict(X_test)
        
        accuracy = accuracy_score(y_test, y_pred)
        bal_accuracy = balanced_accuracy_score(y_test, y_pred)
        threshold = stump.tree_.threshold[0]
        
        if verbose:
            print(f"Accuracy: {accuracy:.6f}")
            print(f"Balanced Accuracy: {bal_accuracy:.6f}")
            print(f"Learned threshold: {threshold:.6f}")
        
        return {
            'accuracy': accuracy,
            'balanced_accuracy': bal_accuracy,
            'threshold': threshold,
            'model': stump,
            'name': 'Cubic (Degree 3)'
        }
    
    def demonstrate_quartic(self, n_samples=12000, test_ensembles=True, verbose=True):
        """
        Demonstrate quartic classification using classical invariants.
        """
        if verbose:
            print("\n" + "="*80)
            print("Quartic demonstration")
            print("="*80)
        
        # Generate data
        A = self.rng.uniform(-10, 10, n_samples)
        B = self.rng.uniform(-10, 10, n_samples)
        C = self.rng.uniform(-10, 10, n_samples)
        D = self.rng.uniform(-10, 10, n_samples)
        
        # Compute classical quartic invariants (for monic: a=1, b=A, c=B, d=C, e=D)
        def quartic_invariants(a, b, c, d, e):
            I = 12*a*e - 3*b*d + c*c
            J = 72*a*c*e + 9*b*c*d - 27*a*d*d - 27*b*b*e - 2*c**3
            Delta_expr = 4*I**3 - J**2
            P = 8*a*c - 3*b*b
            D_aux = 64*a**3*e - 16*a*a*c*c + 16*a*b*b*c - 16*a*a*b*d - 3*b**4
            return I, J, Delta_expr, P, D_aux
        
        I, J, Delta_expr, P, D_aux = quartic_invariants(1.0, A, B, C, D)
        
        # Ground truth via root computation
        def label_quartic(A, B, C, D):
            roots = np.roots([1.0, A, B, C, D])
            k = int(np.sum(np.abs(roots.imag) < 1e-10))
            if k >= 3: return 0  # 4 real
            if k >= 1: return 1  # 2 real
            return 2  # 0 real
        
        y = np.array([label_quartic(A[i], B[i], C[i], D[i]) for i in range(n_samples)])
        
        # Prepare features
        X = np.column_stack([Delta_expr, P, D_aux])
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=self.random_state, stratify=y
        )
        
        results = {}
        
        # Decision Tree
        tree = DecisionTreeClassifier(max_depth=2, random_state=self.random_state)
        tree.fit(X_train, y_train)
        y_pred = tree.predict(X_test)
        
        tree_acc = accuracy_score(y_test, y_pred)
        tree_bal = balanced_accuracy_score(y_test, y_pred)
        
        if verbose:
            print(f"\nDecision Tree:")
            print(f"  Accuracy: {tree_acc:.6f}")
            print(f"  Balanced Accuracy: {tree_bal:.6f}")
        
        results['decision_tree'] = {
            'accuracy': tree_acc,
            'balanced_accuracy': tree_bal,
            'model': tree
        }
        
        # Ensemble Methods
        if test_ensembles:
            models = {
                'Random Forest': RandomForestClassifier(
                    n_estimators=100, 
                    max_depth=10, 
                    random_state=self.random_state
                ),
                'Gradient Boosting': GradientBoostingClassifier(
                    n_estimators=100, 
                    random_state=self.random_state
                )
            }
            
            if verbose:
                print(f"\nEnsemble Methods:")
            
            for name, model in models.items():
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                
                acc = accuracy_score(y_test, y_pred)
                bal_acc = balanced_accuracy_score(y_test, y_pred)
                
                if verbose:
                    print(f"  {name}: Acc={acc:.6f}, Balanced={bal_acc:.6f}")
                
                results[name.lower().replace(' ', '_')] = {
                    'accuracy': acc,
                    'balanced_accuracy': bal_acc,
                    'model': model
                }
        
        results['name'] = 'Quartic'
        return results
    
    # Data generations
    def generate_polynomial_data(self, degree, n_samples, coef_range=(-10, 10)):
        """
        Generate polynomial data for any degree 2-5.
        """
        if degree == 2:
            return self._generate_quadratic_data(n_samples, coef_range)
        elif degree == 3:
            return self._generate_cubic_data(n_samples, coef_range)
        elif degree == 4:
            return self._generate_quartic_data(n_samples, coef_range)
        elif degree == 5:
            return self._generate_quintic_data(n_samples, coef_range)
        else:
            raise ValueError(f"Unsupported degree: {degree}")
    
    def _generate_quadratic_data(self, n_samples, coef_range):
        """Binary classification: real vs complex roots."""
        a = self.rng.uniform(*coef_range, n_samples)
        b = self.rng.uniform(*coef_range, n_samples)
        c = self.rng.uniform(*coef_range, n_samples)
        
        coefficients = np.column_stack([a, b, c])
        discriminant = b**2 - 4*a*c
        labels = (discriminant < 0).astype(int)
        
        return coefficients, labels, {'discriminant': discriminant}
    
    def _generate_cubic_data(self, n_samples, coef_range):
        """Binary classification: 3 real vs complex pair."""
        A = self.rng.uniform(*coef_range, n_samples)
        B = self.rng.uniform(*coef_range, n_samples)
        C = self.rng.uniform(*coef_range, n_samples)
        
        coefficients = np.column_stack([A, B, C])
        labels = np.zeros(n_samples, dtype=int)
        
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i]])
            has_complex = np.any(np.abs(roots.imag) > 1e-10)
            labels[i] = 1 if has_complex else 0
        
        return coefficients, labels, {}
    
    def _generate_quartic_data(self, n_samples, coef_range):
        """3-class: 4 real, 2 real, 0 real."""
        A = self.rng.uniform(*coef_range, n_samples)
        B = self.rng.uniform(*coef_range, n_samples)
        C = self.rng.uniform(*coef_range, n_samples)
        D = self.rng.uniform(*coef_range, n_samples)
        
        coefficients = np.column_stack([A, B, C, D])
        labels = np.zeros(n_samples, dtype=int)
        
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i], D[i]])
            n_real = np.sum(np.abs(roots.imag) < 1e-10)
            if n_real >= 3:
                labels[i] = 0  # 4 or 3 real
            elif n_real >= 1:
                labels[i] = 1  # 2 or 1 real
            else:
                labels[i] = 2  # 0 real
        
        return coefficients, labels, {}
    
    def _generate_quintic_data(self, n_samples, coef_range):
        """3-class: 5 real, 3 real, 1 real."""
        A = self.rng.uniform(*coef_range, n_samples)
        B = self.rng.uniform(*coef_range, n_samples)
        C = self.rng.uniform(*coef_range, n_samples)
        D = self.rng.uniform(*coef_range, n_samples)
        E = self.rng.uniform(*coef_range, n_samples)
        
        coefficients = np.column_stack([A, B, C, D, E])
        labels = np.zeros(n_samples, dtype=int)
        
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i], D[i], E[i]])
            n_real = np.sum(np.abs(roots.imag) < 1e-10)
            if n_real == 5:
                labels[i] = 0
            elif n_real == 3:
                labels[i] = 1
            else:  # n_real == 1
                labels[i] = 2
        
        return coefficients, labels, {}
    
    # Validation tests
    def test_extrapolation(self, degree, model_name='NeuralNetwork', verbose=True):
        """
        Test model generalization to larger coefficient ranges.
        """
        if verbose:
            print(f"\n{'='*60}")
            print(f"EXTRAPOLATION TEST - Degree {degree}")
            print(f"{'='*60}")
        
        # Train on [-10, 10]
        X_train, y_train, _ = self.generate_polynomial_data(degree, 5000, (-10, 10))
        
        # Get model
        model, scaler = self._get_model(model_name, degree)
        
        # Train
        if scaler:
            X_train_scaled = scaler.fit_transform(X_train)
            model.fit(X_train_scaled, y_train)
        else:
            model.fit(X_train, y_train)
        
        # Test on increasing ranges
        test_ranges = [10, 20, 50, 100]
        results = []
        
        for r in test_ranges:
            X_test, y_test, _ = self.generate_polynomial_data(degree, 1000, (-r, r))
            
            if scaler:
                X_test_scaled = scaler.transform(X_test)
                y_pred = model.predict(X_test_scaled)
            else:
                y_pred = model.predict(X_test)
            
            acc = balanced_accuracy_score(y_test, y_pred)
            results.append({'range': r, 'accuracy': acc})
            
            if verbose:
                print(f"  Range ±{r:3d}: {acc:.3f}")
        
        degradation = results[0]['accuracy'] - results[-1]['accuracy']
        if verbose:
            if degradation > 0.1:
                print(f" Degraded by {degradation:.3f}")
            else:
                print(f" Stable (degradation: {degradation:.3f})")
        
        return results
    
    def test_minimal_training(self, degree, model_name='NeuralNetwork', verbose=True):
        """
        Test learning from minimal data.
        """
        if verbose:
            print(f"\n{'='*60}")
            print(f"MINIMAL TRAINING TEST - Degree {degree}")
            print(f"{'='*60}")
        
        training_sizes = [10, 50, 100, 500, 1000, 5000]
        results = []
        
        # Generate fixed test set
        X_test, y_test, _ = self.generate_polynomial_data(degree, 1000, (-10, 10))
        
        for n_train in training_sizes:
            X_train, y_train, _ = self.generate_polynomial_data(degree, n_train, (-10, 10))
            
            model, scaler = self._get_model(model_name, degree)
            
            try:
                if scaler:
                    X_train_scaled = scaler.fit_transform(X_train)
                    X_test_scaled = scaler.transform(X_test)
                    model.fit(X_train_scaled, y_train)
                    y_pred = model.predict(X_test_scaled)
                else:
                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_test)
                
                acc = balanced_accuracy_score(y_test, y_pred)
            except:
                acc = 0.33 if degree >= 4 else 0.5
            
            results.append({'n_samples': n_train, 'accuracy': acc})
            
            if verbose:
                print(f"  {n_train:4d} samples: {acc:.3f}")
        
        return results
    
    def test_noise_robustness(self, degree, model_name='NeuralNetwork', verbose=True):
        """
        Test robustness to noisy test data.
        """
        if verbose:
            print(f"\n{'='*60}")
            print(f"NOISE ROBUSTNESS TEST - Degree {degree}")
            print(f"{'='*60}")
        
        # Generate data
        X, y, _ = self.generate_polynomial_data(degree, 6000, (-10, 10))
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=self.random_state, stratify=y
        )
        
        # Train model
        model, scaler = self._get_model(model_name, degree)
        
        if scaler:
            X_train_scaled = scaler.fit_transform(X_train)
            model.fit(X_train_scaled, y_train)
        else:
            model.fit(X_train, y_train)
        
        # Test with increasing noise
        noise_levels = [0.0, 0.1, 0.5, 1.0, 2.0]
        results = []
        
        if verbose:
            print(f"  ", end="")
        
        for noise in noise_levels:
            X_noisy = X_test + self.rng.normal(0, noise, X_test.shape)
            
            if scaler:
                X_noisy_scaled = scaler.transform(X_noisy)
                y_pred = model.predict(X_noisy_scaled)
            else:
                y_pred = model.predict(X_noisy)
            
            acc = balanced_accuracy_score(y_test, y_pred)
            results.append({'noise': noise, 'accuracy': acc})
            
            if verbose:
                print(f"σ={noise:.1f}:{acc:.3f} ", end="")
        
        degradation = results[0]['accuracy'] - results[-1]['accuracy']
        if verbose:
            print(f" | Degradation: {degradation:.3f}")
        
        return results
    
    def _get_model(self, model_name, degree):
        """Helper to get model with appropriate parameters."""
        scaler = None
        
        if model_name == 'NeuralNetwork':
            model = MLPClassifier(
                hidden_layer_sizes=(100, 50), 
                max_iter=500, 
                random_state=self.random_state
            )
            scaler = StandardScaler()
        elif model_name == 'DecisionTree':
            model = DecisionTreeClassifier(
                max_depth=5, 
                random_state=self.random_state
            )
        elif model_name == 'RandomForest':
            model = RandomForestClassifier(
                n_estimators=100, 
                max_depth=10, 
                random_state=self.random_state
            )
        elif model_name == 'GradientBoosting':
            n_classes = 2 if degree <= 3 else 3
            model = GradientBoostingClassifier(
                n_estimators=100, 
                random_state=self.random_state
            )
        elif model_name == 'XGBoost' and XGBOOST_AVAILABLE:
            params = {
                'n_estimators': 100,
                'random_state': self.random_state,
                'use_label_encoder': False,
                'eval_metric': 'logloss'
            }
            if degree >= 4:
                params['objective'] = 'multi:softmax'
                params['eval_metric'] = 'mlogloss'
            model = XGBClassifier(**params)
        else:
            raise ValueError(f"Unknown model: {model_name}")
        
        return model, scaler
    
    # Run the experiments
    def run_all_demonstrations(self):
        """Run all demonstrations for degrees 2-4."""
        print("\n" + "#"*80)
        print("# Degrees 2-4")
        print("#"*80)
        
        results = {}
        results['quadratic'] = self.demonstrate_quadratic()
        results['cubic'] = self.demonstrate_cubic()
        results['quartic'] = self.demonstrate_quartic(test_ensembles=True)
        
        return results
    
    def run_validation_suite(self, degrees=[2, 3, 4, 5], model='NeuralNetwork'):
        """Run complete validation suite across all degrees."""
        print("\n" + "#"*80)
        print(f"# VALIDATION SUITE: Model={model}")
        print("#"*80)
        
        all_results = {}
        
        for degree in degrees:
            print(f"\n{'#'*80}")
            print(f"# DEGREE {degree}")
            print(f"{'#'*80}")
            
            results = {}
            results['extrapolation'] = self.test_extrapolation(degree, model)
            results['minimal_training'] = self.test_minimal_training(degree, model)
            results['noise_robustness'] = self.test_noise_robustness(degree, model)
            
            all_results[f'degree_{degree}'] = results
        
        return all_results


# Main execution
if __name__ == "__main__":
    print("="*80)
    print(" Polynomial classification suite")
    print("="*80)
    
    # Initialize with fixed random state
    suite = PolynomialClassificationSuite(random_state=42)
    
    # Run demonstrations
    demo_results = suite.run_all_demonstrations()
    
    # Run validation suite on neural networks
    validation_results = suite.run_validation_suite(
        degrees=[2, 3, 4, 5], 
        model='NeuralNetwork'
    )
    
    print("\n" + "="*80)
    print(" All tests completed")
    print("="*80)


In [ ]:
"""
QUINTIC FEATURE ENGINEERING
Advanced mathematical features for quintic polynomial classification
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


class QuinticFeatureExtractor:
    """
    Feature extraction for quintic polynomials.
    
    Implements 6 feature groups (63 total features):
    - Base: 5 (A, B, C, D, E)
    - Sturm: 8
    - Descartes: 6
    - Newton: 10
    - Critical: 10 (includes Crit8)
    - Hybrid: 16
    - Decomposition: 8
    """
    
    def __init__(self):
        pass
    
    # Group 1: Strum sequences (8 features)    
    def sturm_sequence(self, coeffs):
        """Compute Sturm sequence via iterated polynomial division."""
        p = np.poly1d(coeffs)
        p_prime = np.polyder(p)
        sturm_seq = [p, p_prime]
        
        max_iterations = 10
        for _ in range(max_iterations):
            if len(sturm_seq[-1].c) == 0:
                break
            
            _, remainder = np.polydiv(sturm_seq[-2], sturm_seq[-1])
            remainder = -remainder
            
            # Clean up near-zero coefficients
            remainder.c = remainder.c[np.abs(remainder.c) > 1e-10]
            
            if len(remainder.c) == 0 or (len(remainder.c) == 1 and abs(remainder.c[0]) < 1e-10):
                break
            
            sturm_seq.append(remainder)
        
        return sturm_seq
    
    def count_sign_changes(self, sturm_seq, x):
        """Count sign changes in Sturm sequence at point x."""
        signs = []
        for poly in sturm_seq:
            val = poly(x)
            if abs(val) > 1e-10:
                signs.append(np.sign(val))
        
        if len(signs) <= 1:
            return 0
        
        changes = sum(1 for i in range(len(signs)-1) if signs[i] * signs[i+1] < 0)
        return changes
    
    def extract_sturm_features(self, coefficients):
        """Extract 8 Sturm-based features."""
        n_samples = len(coefficients)
        sturm_features = np.zeros((n_samples, 8))
        
        for i in range(n_samples):
            try:
                coeffs = np.concatenate([[1], coefficients[i]])
                sturm_seq = self.sturm_sequence(coeffs)
                
                sturm_features[i, 0] = self.count_sign_changes(sturm_seq, -1000)
                sturm_features[i, 1] = self.count_sign_changes(sturm_seq, 1000)
                sturm_features[i, 2] = abs(sturm_features[i, 0] - sturm_features[i, 1])
                
                test_points = [-10, -1, 0, 1, 10]
                for j, x in enumerate(test_points):
                    sturm_features[i, 3+j] = self.count_sign_changes(sturm_seq, x)
            except:
                sturm_features[i, :] = [2, 2, 3, 2, 2, 2, 2, 2]
        
        return sturm_features

    
    # Group 2: Descartes' rule (6 features)
    def extract_descartes_features(self, coefficients):
        """Extract 6 Descartes-based features."""
        n_samples = len(coefficients)
        descartes_features = np.zeros((n_samples, 6))
        
        for i in range(n_samples):
            poly_coeffs = np.concatenate([[1], coefficients[i]])
            
            # Positive roots
            nonzero_coeffs = poly_coeffs[np.abs(poly_coeffs) > 1e-10]
            sign_changes_pos = sum(1 for j in range(len(nonzero_coeffs)-1) 
                                  if nonzero_coeffs[j] * nonzero_coeffs[j+1] < 0)
            
            # Negative roots (substitute x = -x)
            neg_poly_coeffs = poly_coeffs.copy()
            for j in range(len(neg_poly_coeffs)):
                if (len(neg_poly_coeffs) - 1 - j) % 2 == 1:
                    neg_poly_coeffs[j] *= -1
            
            nonzero_neg = neg_poly_coeffs[np.abs(neg_poly_coeffs) > 1e-10]
            sign_changes_neg = sum(1 for j in range(len(nonzero_neg)-1) 
                                  if nonzero_neg[j] * nonzero_neg[j+1] < 0)
            
            descartes_features[i, 0] = sign_changes_pos
            descartes_features[i, 1] = sign_changes_neg
            descartes_features[i, 2] = sign_changes_pos + sign_changes_neg
            descartes_features[i, 3] = 5 - descartes_features[i, 2]
            descartes_features[i, 4] = sign_changes_pos % 2
            descartes_features[i, 5] = sign_changes_neg % 2
        
        return descartes_features
    
    # Group 3: nNewton's sums (10 features)    
    def compute_newton_sums(self, A, B, C, D, E):
        """Compute Newton's sums s1-s5 via Newton's identities."""
        s1 = -A
        s2 = A**2 - 2*B
        s3 = -A**3 + 3*A*B - 3*C
        s4 = A**4 - 4*A**2*B + 2*B**2 + 4*A*C - 4*D
        s5 = -A**5 + 5*A**3*B - 5*A*B**2 - 5*A**2*C + 5*B*C + 5*A*D - 5*E
        return s1, s2, s3, s4, s5
    
    def extract_newton_features(self, coefficients):
        """Extract 10 Newton-based features."""
        n_samples = len(coefficients)
        newton_features = np.zeros((n_samples, 10))
        
        for i in range(n_samples):
            A, B, C, D, E = coefficients[i]
            s1, s2, s3, s4, s5 = self.compute_newton_sums(A, B, C, D, E)
            
            newton_features[i, 0:5] = [s1, s2, s3, s4, s5]
            newton_features[i, 5] = s1 / 5
            newton_features[i, 6] = s2/5 - (s1/5)**2
            newton_features[i, 7] = s3 / (abs(s1) + 1)
            newton_features[i, 8] = s4 / (abs(s2) + 1)
            newton_features[i, 9] = s5 / (abs(s3) + 1)
        
        return newton_features
    
    # Group 4: Critical points (10 features, includes Crit8)    
    def extract_critical_point_features(self, coefficients):
        """Extract 10 critical point features including Crit8."""
        n_samples = len(coefficients)
        critical_features = np.zeros((n_samples, 10))
        
        for i in range(n_samples):
            A, B, C, D, E = coefficients[i]
            deriv_coeffs = [5, 4*A, 3*B, 2*C, D]
            
            try:
                critical_points = np.roots(deriv_coeffs)
                real_critical = critical_points[np.abs(critical_points.imag) < 1e-10].real
                
                critical_features[i, 0] = len(real_critical)
                
                if len(real_critical) > 0:
                    critical_features[i, 1] = np.min(real_critical)
                    critical_features[i, 2] = np.max(real_critical)
                    critical_features[i, 3] = np.mean(real_critical)
                    critical_features[i, 4] = np.std(real_critical) if len(real_critical) > 1 else 0
                    
                    poly = np.poly1d([1, A, B, C, D, E])
                    critical_values = [poly(x) for x in real_critical]
                    
                    critical_features[i, 5] = np.min(critical_values)
                    critical_features[i, 6] = np.max(critical_values)
                    critical_features[i, 7] = np.mean(critical_values)
                    
                    # CRIT8: Sign changes at critical points
                    nonzero_vals = [v for v in critical_values if abs(v) > 1e-10]
                    if len(nonzero_vals) > 1:
                        sign_changes = sum(1 for j in range(len(nonzero_vals)-1) 
                                         if nonzero_vals[j] * nonzero_vals[j+1] < 0)
                        critical_features[i, 8] = sign_changes
                
                # Inflection points
                second_deriv = [20, 12*A, 6*B, 2*C]
                inflection_roots = np.roots(second_deriv)
                real_inflections = inflection_roots[np.abs(inflection_roots.imag) < 1e-10]
                critical_features[i, 9] = len(real_inflections)
            except:
                critical_features[i, :] = 0
        
        return critical_features
    
    # Group 5: Hybrid symbolic (16 features)
    def create_hybrid_features(self, coefficients):
        """Extract 16 hybrid algebraic features."""
        n_samples = len(coefficients)
        hybrid_features = []
        
        for i in range(n_samples):
            A, B, C, D, E = coefficients[i]
            
            # Tschirnhaus invariants
            I2 = A**2 - 2*B
            I3 = A**3 - 3*A*B + 3*C
            I4 = A**4 - 4*A**2*B + 2*B**2 + 4*A*C - 4*D
            I5 = A**5 - 5*A**3*B + 5*A*B**2 + 5*A**2*C - 5*B*C - 5*A*D + 5*E
            
            # Novel combinations
            S1 = A*B*C - D*E
            S2 = A**2*E - B**2*D + C**3
            S3 = (A*D - B*C)**2 + (B*E - C*D)**2
            S4 = A*C*E - B*D**2
            
            # Discriminant-like differences
            D1 = B**2 - A*C
            D2 = C**2 - B*D
            D3 = D**2 - C*E
            
            # Scale-invariant ratios
            eps = 1e-10
            R1 = (A*E) / (B*D + eps)
            R2 = (B*D) / (C**2 + eps)
            R3 = (A*C*E) / (B*D**2 + eps)
            R4 = I2 / (I3 + eps)
            R5 = I3 / (I4 + eps)
            
            features = [I2, I3, I4, I5, S1, S2, S3, S4, D1, D2, D3, R1, R2, R3, R4, R5]
            hybrid_features.append(features)
        
        return np.array(hybrid_features)
    
    # Group 6: Decomposition (8 features)    
    def extract_decomposition_features(self, coefficients):
        """Extract 8 decomposition/factorization features."""
        n_samples = len(coefficients)
        decomp_features = np.zeros((n_samples, 8))
        
        for i in range(n_samples):
            A, B, C, D, E = coefficients[i]
            coeffs = [A, B, C, D, E]
            
            decomp_features[i, 0] = np.sum(np.abs(coeffs) < 0.1)
            
            abs_coeffs = np.abs(coeffs)
            decomp_features[i, 1] = np.max(abs_coeffs) / (np.min(abs_coeffs) + 1e-10)
            decomp_features[i, 2] = np.var(abs_coeffs)
            decomp_features[i, 3] = abs(E) < 0.1
            decomp_features[i, 4] = abs(A*E - B*D + C**2) / (abs(A*E) + abs(B*D) + abs(C**2) + 1)
            decomp_features[i, 5] = abs(A**2 - B) / (abs(A**2) + abs(B) + 1)
            decomp_features[i, 6] = abs(B**2 - A*C) / (abs(B**2) + abs(A*C) + 1)
            decomp_features[i, 7] = abs(C**2 - B*D) / (abs(C**2) + abs(B*D) + 1)
        
        return decomp_features
    

    # Main extraction
    def extract_all_features(self, coefficients):
        """Extract all 63 features."""
        sturm_features = self.extract_sturm_features(coefficients)
        descartes_features = self.extract_descartes_features(coefficients)
        newton_features = self.extract_newton_features(coefficients)
        critical_features = self.extract_critical_point_features(coefficients)
        hybrid_features = self.create_hybrid_features(coefficients)
        decomp_features = self.extract_decomposition_features(coefficients)
        
        all_features = np.hstack([
            coefficients,
            sturm_features,
            descartes_features,
            newton_features,
            critical_features,
            hybrid_features,
            decomp_features
        ])
        
        return all_features


class QuinticFeatureComparison:
    """
    Compare baseline vs individual methods vs combined features.
    """
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.rng = np.random.RandomState(random_state)
        self.feature_extractor = QuinticFeatureExtractor()
    
    def generate_quintic_data(self, n_samples, coef_range=(-10, 10)):
        """Generate quintic data with proper RNG."""
        A = self.rng.uniform(*coef_range, n_samples)
        B = self.rng.uniform(*coef_range, n_samples)
        C = self.rng.uniform(*coef_range, n_samples)
        D = self.rng.uniform(*coef_range, n_samples)
        E = self.rng.uniform(*coef_range, n_samples)
        
        coefficients = np.column_stack([A, B, C, D, E])
        labels = np.zeros(n_samples, dtype=int)
        
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i], D[i], E[i]])
            n_real = np.sum(np.abs(roots.imag) < 1e-10)
            
            if n_real == 5:
                labels[i] = 0
            elif n_real == 3:
                labels[i] = 1
            else:
                labels[i] = 2
        
        return coefficients, labels
    
    def test_baseline_single_trial(self, n_samples=5000, verbose=True):
        """Single trial baseline."""
        if verbose:
            print("\n" + "="*60)
            print("Baseline: Raw Coefficients Only (Single Trial)")
            print("="*60)
        
        coefficients, y = self.generate_quintic_data(n_samples)
        
        X_train, X_test, y_train, y_test = train_test_split(
            coefficients, y, test_size=0.2, random_state=42, stratify=y
        )
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        model = MLPClassifier(
            hidden_layer_sizes=(100, 50),
            max_iter=500,
            random_state=42
        )
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        
        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        
        if verbose:
            print(f"Neural Network: Acc={acc:.3f}, Balanced={bal_acc:.3f}")
        
        return bal_acc
    
    def test_baseline_multi_trial(self, n_samples=5000, n_trials=20, verbose=True):
        """
        20-trial baseline validation.
        """
        if verbose:
            print("\n" + "="*60)
            print(f"Baseline: Raw Coefficients Only ({n_trials} trials)")
            print("="*60)
        
        accuracies = []
        
        for trial in range(n_trials):
            # Create new RNG for each trial
            trial_rng = np.random.RandomState(trial)
            
            # Generate data
            A = trial_rng.uniform(-10, 10, n_samples)
            B = trial_rng.uniform(-10, 10, n_samples)
            C = trial_rng.uniform(-10, 10, n_samples)
            D = trial_rng.uniform(-10, 10, n_samples)
            E = trial_rng.uniform(-10, 10, n_samples)
            
            coefficients = np.column_stack([A, B, C, D, E])
            labels = np.zeros(n_samples, dtype=int)
            
            for i in range(n_samples):
                roots = np.roots([1, A[i], B[i], C[i], D[i], E[i]])
                n_real = np.sum(np.abs(roots.imag) < 1e-10)
                
                if n_real == 5:
                    labels[i] = 0
                elif n_real == 3:
                    labels[i] = 1
                else:
                    labels[i] = 2
            
            # Train-test split
            X_train, X_test, y_train, y_test = train_test_split(
                coefficients, labels, test_size=0.2, random_state=42, stratify=labels
            )
            
            # Train model
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            model = MLPClassifier(
                hidden_layer_sizes=(100, 50),
                max_iter=500,
                random_state=42
            )
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            
            bal_acc = balanced_accuracy_score(y_test, y_pred)
            accuracies.append(bal_acc)
            
            if verbose and (trial < 3 or trial == n_trials-1):
                print(f"  Trial {trial+1:2d}: {bal_acc:.3f}")
        
        mean_acc = np.mean(accuracies)
        std_acc = np.std(accuracies)
        
        if verbose:
            print(f"\nMean: {mean_acc:.3f} ± {std_acc:.3f}")
            print(f"Range: [{np.min(accuracies):.3f}, {np.max(accuracies):.3f}]")
        
        return mean_acc, std_acc, accuracies
    
    def test_individual_methods(self, n_samples=5000, verbose=True):
        """Test each feature group individually."""
        if verbose:
            print("\n" + "="*60)
            print("Individual method testing")
            print("="*60)
        
        coefficients, y = self.generate_quintic_data(n_samples)
        
        methods = {
            'Sturm': self.feature_extractor.extract_sturm_features(coefficients),
            'Descartes': self.feature_extractor.extract_descartes_features(coefficients),
            'Newton': self.feature_extractor.extract_newton_features(coefficients),
            'Critical': self.feature_extractor.extract_critical_point_features(coefficients),
            'Hybrid': self.feature_extractor.create_hybrid_features(coefficients),
            'Decomposition': self.feature_extractor.extract_decomposition_features(coefficients)
        }
        
        results = {}
        
        for method_name, features in methods.items():
            X = np.hstack([coefficients, features])
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y
            )
            
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            model = MLPClassifier(
                hidden_layer_sizes=(100, 50),
                max_iter=500,
                random_state=42
            )
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            
            acc = accuracy_score(y_test, y_pred)
            bal_acc = balanced_accuracy_score(y_test, y_pred)
            
            results[method_name] = {
                'accuracy': acc,
                'balanced_accuracy': bal_acc
            }
            
            if verbose:
                print(f"  {method_name:15s}: Acc={acc:.3f}, Balanced={bal_acc:.3f}")
        
        return results
    
    def test_individual_methods_multi_trial(self, n_samples=5000, n_trials=20, verbose=True):
        """
        Test each feature group individually across 20 trials.
        """
        if verbose:
            print("\n" + "="*60)
            print(f"Individual method testing ({n_trials} trials)")
            print("="*60)
        
        method_names = ['Sturm', 'Descartes', 'Newton', 'Critical', 'Hybrid', 'Decomposition']
        method_accuracies = {name: [] for name in method_names}
        
        for trial in range(n_trials):
            # Create new RNG for each trial
            trial_rng = np.random.RandomState(trial)
            
            # Generate data
            A = trial_rng.uniform(-10, 10, n_samples)
            B = trial_rng.uniform(-10, 10, n_samples)
            C = trial_rng.uniform(-10, 10, n_samples)
            D = trial_rng.uniform(-10, 10, n_samples)
            E = trial_rng.uniform(-10, 10, n_samples)
            
            coefficients = np.column_stack([A, B, C, D, E])
            labels = np.zeros(n_samples, dtype=int)
            
            for i in range(n_samples):
                roots = np.roots([1, A[i], B[i], C[i], D[i], E[i]])
                n_real = np.sum(np.abs(roots.imag) < 1e-10)
                
                if n_real == 5:
                    labels[i] = 0
                elif n_real == 3:
                    labels[i] = 1
                else:
                    labels[i] = 2
            
            # Extract features
            methods = {
                'Sturm': self.feature_extractor.extract_sturm_features(coefficients),
                'Descartes': self.feature_extractor.extract_descartes_features(coefficients),
                'Newton': self.feature_extractor.extract_newton_features(coefficients),
                'Critical': self.feature_extractor.extract_critical_point_features(coefficients),
                'Hybrid': self.feature_extractor.create_hybrid_features(coefficients),
                'Decomposition': self.feature_extractor.extract_decomposition_features(coefficients)
            }
            
            for method_name, features in methods.items():
                X = np.hstack([coefficients, features])
                X_train, X_test, y_train, y_test = train_test_split(
                    X, labels, test_size=0.2, random_state=42, stratify=labels
                )
                
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)
                
                model = MLPClassifier(
                    hidden_layer_sizes=(100, 50),
                    max_iter=500,
                    random_state=42
                )
                model.fit(X_train_scaled, y_train)
                y_pred = model.predict(X_test_scaled)
                
                bal_acc = balanced_accuracy_score(y_test, y_pred)
                method_accuracies[method_name].append(bal_acc)
            
            if verbose and (trial < 3 or trial == n_trials-1):
                print(f"  Trial {trial+1:2d}: ", end="")
                for name in method_names:
                    print(f"{name[:4]}={method_accuracies[name][-1]:.3f} ", end="")
                print()
        
        # Calculate statistics
        results = {}
        if verbose:
            print(f"\nMean ± Std (across {n_trials} trials):")
        
        for method_name in method_names:
            accs = method_accuracies[method_name]
            mean_acc = np.mean(accs)
            std_acc = np.std(accs)
            
            results[method_name] = {
                'mean': mean_acc,
                'std': std_acc,
                'all_accuracies': accs
            }
            
            if verbose:
                print(f"  {method_name:15s}: {mean_acc:.3f} ± {std_acc:.3f}")
        
        return results
    
    def test_combined(self, n_samples=5000, verbose=True):
        """Test all 63 features combined."""
        if verbose:
            print("\n" + "="*60)
            print("Combined: All 63 Features")
            print("="*60)
        
        coefficients, y = self.generate_quintic_data(n_samples)
        X = self.feature_extractor.extract_all_features(coefficients)
        
        if verbose:
            print(f"Total features: {X.shape[1]}")
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        model = MLPClassifier(
            hidden_layer_sizes=(200, 100, 50),
            max_iter=500,
            random_state=42
        )
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        
        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        
        if verbose:
            print(f"Neural Network: Acc={acc:.3f}, Balanced={bal_acc:.3f}")
        
        return {'accuracy': acc, 'balanced_accuracy': bal_acc}
    
    def run_comprehensive_analysis(self, multi_trial_methods=False):
        """Run all tests."""
        print("="*80)
        print(" Quintic Feature Engineering: Full Test")
        print("="*80)
        
        # 1. Single trial baseline
        single_baseline = self.test_baseline_single_trial()
        
        # 2. Multi-trial baseline
        multi_baseline_mean, multi_baseline_std, _ = self.test_baseline_multi_trial()
        
        # 3. Individual methods
        if multi_trial_methods:
            individual_results = self.test_individual_methods_multi_trial()
        else:
            individual_results = self.test_individual_methods()
        
        # 4. Combined features
        combined_results = self.test_combined()
        
        print("\n" + "="*80)
        print(" SUMMARY")
        print("="*80)
        print(f"Baseline (single trial): {single_baseline:.3f}")
        print(f"Baseline (20 trials):    {multi_baseline_mean:.3f} ± {multi_baseline_std:.3f}")
        
        if multi_trial_methods:
            print(f"\nIndividual Methods (20 trials):")
            for method, result in individual_results.items():
                print(f"  {method:15s}: {result['mean']:.3f} ± {result['std']:.3f}")
        
        print(f"\nCombined (63 features):  {combined_results['balanced_accuracy']:.3f}")
        
        return {
            'baseline_single': single_baseline,
            'baseline_multi_mean': multi_baseline_mean,
            'baseline_multi_std': multi_baseline_std,
            'individual': individual_results,
            'combined': combined_results
        }



# Main execution
if __name__ == "__main__":
    comparison = QuinticFeatureComparison(random_state=42)
    
    results = comparison.run_comprehensive_analysis(multi_trial_methods=True)


In [ ]:
"""
QUINTIC RULE EXTRACTION
"""

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, balanced_accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Feature extractor
class QuinticFeatureExtractor:
    def sturm_sequence(self, coeffs):
        p = np.poly1d(coeffs)
        p_prime = np.polyder(p)
        sturm_seq = [p, p_prime]
        for _ in range(10):
            if len(sturm_seq[-1].c) == 0: break
            _, remainder = np.polydiv(sturm_seq[-2], sturm_seq[-1])
            remainder = -remainder
            remainder.c = remainder.c[np.abs(remainder.c) > 1e-10]
            if len(remainder.c) == 0: break
            sturm_seq.append(remainder)
        return sturm_seq
    
    def count_sign_changes(self, sturm_seq, x):
        signs = [np.sign(poly(x)) for poly in sturm_seq if abs(poly(x)) > 1e-10]
        return sum(1 for i in range(len(signs)-1) if signs[i] * signs[i+1] < 0)
    
    def extract_all_features(self, coefficients):
        n = len(coefficients)
        all_features = [coefficients]
        
        # Sturm (8)
        sturm = np.zeros((n, 8))
        for i in range(n):
            try:
                seq = self.sturm_sequence(np.concatenate([[1], coefficients[i]]))
                sturm[i, 0] = self.count_sign_changes(seq, -1000)
                sturm[i, 1] = self.count_sign_changes(seq, 1000)
                sturm[i, 2] = abs(sturm[i, 0] - sturm[i, 1])
                for j, x in enumerate([-10, -1, 0, 1, 10]):
                    sturm[i, 3+j] = self.count_sign_changes(seq, x)
            except: sturm[i, :] = [2, 2, 3, 2, 2, 2, 2, 2]
        all_features.append(sturm)
        
        # Descartes (6)
        descartes = np.zeros((n, 6))
        for i in range(n):
            poly = np.concatenate([[1], coefficients[i]])
            nz = poly[np.abs(poly) > 1e-10]
            sc_pos = sum(1 for j in range(len(nz)-1) if nz[j] * nz[j+1] < 0)
            neg = poly.copy()
            for j in range(len(neg)):
                if (len(neg)-1-j) % 2 == 1: neg[j] *= -1
            nz_neg = neg[np.abs(neg) > 1e-10]
            sc_neg = sum(1 for j in range(len(nz_neg)-1) if nz_neg[j] * nz_neg[j+1] < 0)
            descartes[i] = [sc_pos, sc_neg, sc_pos+sc_neg, 5-(sc_pos+sc_neg), sc_pos%2, sc_neg%2]
        all_features.append(descartes)
        
        # Newton (10)
        newton = np.zeros((n, 10))
        for i in range(n):
            A, B, C, D, E = coefficients[i]
            s1 = -A
            s2 = A**2 - 2*B
            s3 = -A**3 + 3*A*B - 3*C
            s4 = A**4 - 4*A**2*B + 2*B**2 + 4*A*C - 4*D
            s5 = -A**5 + 5*A**3*B - 5*A*B**2 - 5*A**2*C + 5*B*C + 5*A*D - 5*E
            newton[i] = [s1, s2, s3, s4, s5, s1/5, s2/5-(s1/5)**2, 
                        s3/(abs(s1)+1), s4/(abs(s2)+1), s5/(abs(s3)+1)]
        all_features.append(newton)
        
        # Critical points (10) 
        critical = np.zeros((n, 10))
        for i in range(n):
            A, B, C, D, E = coefficients[i]
            try:
                cpts = np.roots([5, 4*A, 3*B, 2*C, D])
                real_cpts = cpts[np.abs(cpts.imag) < 1e-10].real
                critical[i, 0] = len(real_cpts)
                if len(real_cpts) > 0:
                    critical[i, 1:5] = [np.min(real_cpts), np.max(real_cpts), 
                                       np.mean(real_cpts), np.std(real_cpts) if len(real_cpts)>1 else 0]
                    poly = np.poly1d([1, A, B, C, D, E])
                    vals = [poly(x) for x in real_cpts]
                    critical[i, 5:8] = [np.min(vals), np.max(vals), np.mean(vals)]
                    nz_vals = [v for v in vals if abs(v) > 1e-10]
                    if len(nz_vals) > 1:
                        critical[i, 8] = sum(1 for j in range(len(nz_vals)-1) 
                                           if nz_vals[j] * nz_vals[j+1] < 0)  # CRIT8!
                infl = np.roots([20, 12*A, 6*B, 2*C])
                critical[i, 9] = len(infl[np.abs(infl.imag) < 1e-10])
            except: pass
        all_features.append(critical)
        
        # Hybrid (16)
        hybrid = []
        for i in range(n):
            A, B, C, D, E = coefficients[i]
            I2 = A**2 - 2*B
            I3 = A**3 - 3*A*B + 3*C
            I4 = A**4 - 4*A**2*B + 2*B**2 + 4*A*C - 4*D
            I5 = A**5 - 5*A**3*B + 5*A*B**2 + 5*A**2*C - 5*B*C - 5*A*D + 5*E
            eps = 1e-10
            hybrid.append([I2, I3, I4, I5, A*B*C-D*E, A**2*E-B**2*D+C**3,
                          (A*D-B*C)**2+(B*E-C*D)**2, A*C*E-B*D**2,
                          B**2-A*C, C**2-B*D, D**2-C*E,
                          (A*E)/(B*D+eps), (B*D)/(C**2+eps), (A*C*E)/(B*D**2+eps),
                          I2/(I3+eps), I3/(I4+eps)])
        all_features.append(np.array(hybrid))
        
        # Decomposition (8)
        decomp = np.zeros((n, 8))
        for i in range(n):
            A, B, C, D, E = coefficients[i]
            cs = [A, B, C, D, E]
            abs_cs = np.abs(cs)
            decomp[i] = [np.sum(abs_cs < 0.1), np.max(abs_cs)/(np.min(abs_cs)+1e-10),
                        np.var(abs_cs), abs(E)<0.1,
                        abs(A*E-B*D+C**2)/(abs(A*E)+abs(B*D)+abs(C**2)+1),
                        abs(A**2-B)/(abs(A**2)+abs(B)+1),
                        abs(B**2-A*C)/(abs(B**2)+abs(A*C)+1),
                        abs(C**2-B*D)/(abs(C**2)+abs(B*D)+1)]
        all_features.append(decomp)
        
        return np.hstack(all_features)

# Rule extractor
class QuinticRuleExtractor:
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.rng = np.random.RandomState(random_state)
        self.feature_extractor = QuinticFeatureExtractor()
        self.model = None
        self.scaler = None
    
    def generate_quintic_data(self, n_samples):
        A = self.rng.uniform(-10, 10, n_samples)
        B = self.rng.uniform(-10, 10, n_samples)
        C = self.rng.uniform(-10, 10, n_samples)
        D = self.rng.uniform(-10, 10, n_samples)
        E = self.rng.uniform(-10, 10, n_samples)
        coefficients = np.column_stack([A, B, C, D, E])
        labels = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            roots = np.roots([1, A[i], B[i], C[i], D[i], E[i]])
            n_real = np.sum(np.abs(roots.imag) < 1e-10)
            labels[i] = 0 if n_real == 5 else (1 if n_real == 3 else 2)
        return self.feature_extractor.extract_all_features(coefficients), labels
    
    def test_crit8_alone(self, n_samples=5000, verbose=True):
        """Test Crit8 performance with just base coefficients."""
        if verbose:
            print("\n" + "="*80)
            print("Crit8 standalone test.")
            print("="*80)
        
        X_all, y = self.generate_quintic_data(n_samples)
        X_base = X_all[:, :5]
        crit8 = X_all[:, 37].reshape(-1, 1)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X_base, y, test_size=0.2, random_state=42, stratify=y
        )
        
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc = scaler.transform(X_test)
        
        nn_baseline = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
        nn_baseline.fit(X_train_sc, y_train)
        baseline_acc = balanced_accuracy_score(y_test, nn_baseline.predict(X_test_sc))
        
        if verbose:
            print(f"\nBaseline (5 coefficients only):")
            print(f"  Balanced Accuracy: {baseline_acc:.3f}")
        
        X_with_crit8 = np.hstack([X_base, crit8])
        X_train, X_test, y_train, y_test = train_test_split(
            X_with_crit8, y, test_size=0.2, random_state=42, stratify=y
        )
        
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc = scaler.transform(X_test)
        
        nn_crit8 = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
        nn_crit8.fit(X_train_sc, y_train)
        crit8_acc = balanced_accuracy_score(y_test, nn_crit8.predict(X_test_sc))
        improvement = crit8_acc - baseline_acc
        
        if verbose:
            print(f"\nWith Crit8 (5 coefficients + Crit8):")
            print(f"  Balanced Accuracy: {crit8_acc:.3f}")
            print(f"  Improvement: +{improvement:.3f} ({improvement/baseline_acc*100:.1f}%)")
            print(f"\n*** Adding Crit8 alone improves from {baseline_acc:.1%} to {crit8_acc:.1%} ***")
        
        return {'baseline': baseline_acc, 'with_crit8': crit8_acc, 'improvement': improvement}
    
    def test_crit8_alone_multi_trial(self, n_samples=5000, n_trials=20, verbose=True):
        """Test Crit8 across 20 trials - CRITICAL for fair comparison."""
        if verbose:
            print("\n" + "="*80)
            print(f"Crit8 Standalone test ({n_trials} trials)")
            print("="*80)
        
        baseline_accs, crit8_accs = [], []
        
        for trial in range(n_trials):
            trial_extractor = QuinticRuleExtractor(random_state=trial)
            X_all, y = trial_extractor.generate_quintic_data(n_samples)
            X_base = X_all[:, :5]
            crit8 = X_all[:, 37].reshape(-1, 1)
            
            # Baseline
            X_train, X_test, y_train, y_test = train_test_split(
                X_base, y, test_size=0.2, random_state=42, stratify=y
            )
            scaler = StandardScaler()
            X_train_sc = scaler.fit_transform(X_train)
            X_test_sc = scaler.transform(X_test)
            nn_baseline = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
            nn_baseline.fit(X_train_sc, y_train)
            baseline_acc = balanced_accuracy_score(y_test, nn_baseline.predict(X_test_sc))
            baseline_accs.append(baseline_acc)
            
            # With Crit8
            X_with_crit8 = np.hstack([X_base, crit8])
            X_train, X_test, y_train, y_test = train_test_split(
                X_with_crit8, y, test_size=0.2, random_state=42, stratify=y
            )
            scaler = StandardScaler()
            X_train_sc = scaler.fit_transform(X_train)
            X_test_sc = scaler.transform(X_test)
            nn_crit8 = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
            nn_crit8.fit(X_train_sc, y_train)
            crit8_acc = balanced_accuracy_score(y_test, nn_crit8.predict(X_test_sc))
            crit8_accs.append(crit8_acc)
            
            if verbose and (trial < 3 or trial == n_trials-1):
                print(f"  Trial {trial+1:2d}: Baseline={baseline_acc:.3f}, +Crit8={crit8_acc:.3f}, Δ={crit8_acc-baseline_acc:+.3f}")
        
        baseline_mean, baseline_std = np.mean(baseline_accs), np.std(baseline_accs)
        crit8_mean, crit8_std = np.mean(crit8_accs), np.std(crit8_accs)
        improvement_mean = crit8_mean - baseline_mean
        
        if verbose:
            print(f"\nNeural Network Results:")
            print(f"  Baseline: {baseline_mean:.3f} ± {baseline_std:.3f}")
            print(f"  +Crit8:   {crit8_mean:.3f} ± {crit8_std:.3f}")
            print(f"  Improvement: +{improvement_mean:.3f} ({improvement_mean/baseline_mean*100:.1f}%)")
        
        return {
            'baseline_mean': baseline_mean, 'baseline_std': baseline_std,
            'crit8_mean': crit8_mean, 'crit8_std': crit8_std,
            'improvement': improvement_mean,
            'all_baselines': baseline_accs, 'all_crit8': crit8_accs
        }
    
    def test_crit8_decision_tree(self, n_samples=5000, n_trials=20, verbose=True):
        """Test Crit8 with decision trees."""
        if verbose:
            print("\n" + "="*80)
            print(f"Crit8 with decision trees ({n_trials} trials)")
            print("="*80)
        
        dt_baseline_accs, dt_crit8_accs = [], []
        
        for trial in range(n_trials):
            trial_extractor = QuinticRuleExtractor(random_state=trial)
            X_all, y = trial_extractor.generate_quintic_data(n_samples)
            X_base = X_all[:, :5]
            crit8 = X_all[:, 37].reshape(-1, 1)
            
            # Decision tree baseline (coefficients only)
            X_train, X_test, y_train, y_test = train_test_split(
                X_base, y, test_size=0.2, random_state=42, stratify=y
            )
            dt_baseline = DecisionTreeClassifier(max_depth=8, random_state=42)
            dt_baseline.fit(X_train, y_train)
            dt_baseline_acc = balanced_accuracy_score(y_test, dt_baseline.predict(X_test))
            dt_baseline_accs.append(dt_baseline_acc)
            
            # Decision tree with crit8
            X_with_crit8 = np.hstack([X_base, crit8])
            X_train, X_test, y_train, y_test = train_test_split(
                X_with_crit8, y, test_size=0.2, random_state=42, stratify=y
            )
            dt_crit8 = DecisionTreeClassifier(max_depth=8, random_state=42)
            dt_crit8.fit(X_train, y_train)
            dt_crit8_acc = balanced_accuracy_score(y_test, dt_crit8.predict(X_test))
            dt_crit8_accs.append(dt_crit8_acc)
            
            if verbose and (trial < 3 or trial == n_trials-1):
                print(f"  Trial {trial+1:2d}: Baseline={dt_baseline_acc:.3f}, +Crit8={dt_crit8_acc:.3f}, Δ={dt_crit8_acc-dt_baseline_acc:+.3f}")
        
        dt_baseline_mean, dt_baseline_std = np.mean(dt_baseline_accs), np.std(dt_baseline_accs)
        dt_crit8_mean, dt_crit8_std = np.mean(dt_crit8_accs), np.std(dt_crit8_accs)
        dt_improvement_mean = dt_crit8_mean - dt_baseline_mean
        
        if verbose:
            print(f"\nDecision Tree Results:")
            print(f"  Baseline: {dt_baseline_mean:.3f} ± {dt_baseline_std:.3f}")
            print(f"  +Crit8:   {dt_crit8_mean:.3f} ± {dt_crit8_std:.3f}")
            print(f"  Improvement: +{dt_improvement_mean:.3f} ({dt_improvement_mean/dt_baseline_mean*100:.1f}%)")
        
        return {
            'dt_baseline_mean': dt_baseline_mean, 'dt_baseline_std': dt_baseline_std,
            'dt_crit8_mean': dt_crit8_mean, 'dt_crit8_std': dt_crit8_std,
            'dt_improvement': dt_improvement_mean,
            'all_dt_baselines': dt_baseline_accs, 'all_dt_crit8': dt_crit8_accs
        }
    
    def train_and_distill(self, n_samples=5000, verbose=True):
        """Single detailed trial with full analysis."""
        if verbose:
            print("\n" + "="*80)
            print("Single-trial run")
            print("="*80)
        
        X, y = self.generate_quintic_data(n_samples)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        if verbose:
            print(f"\nData: {len(X_train)} train, {len(X_test)} test")
            print(f"Features: {X.shape[1]} (5 base + 58 advanced)")
            print("\nTraining Neural Network")
        
        self.scaler = StandardScaler()
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        self.model = MLPClassifier(hidden_layer_sizes=(200, 100, 50), max_iter=500, random_state=42)
        self.model.fit(X_train_scaled, y_train)
        
        y_pred = self.model.predict(X_test_scaled)
        nn_acc = accuracy_score(y_test, y_pred)
        nn_bal = balanced_accuracy_score(y_test, y_pred)
        
        if verbose:
            print(f"Neural Network Test Performance:")
            print(f"  Accuracy: {nn_acc:.3f}, Balanced Accuracy: {nn_bal:.3f}")
            print("\nDistilling to Decision Tree")
        
        nn_train_pred = self.model.predict(X_train_scaled)
        nn_test_pred = self.model.predict(X_test_scaled)
        tree = DecisionTreeClassifier(max_depth=8, random_state=42)
        tree.fit(X_train, nn_train_pred)
        tree_test_pred = tree.predict(X_test)
        
        test_fidelity = accuracy_score(nn_test_pred, tree_test_pred)
        tree_test_bal = balanced_accuracy_score(y_test, tree_test_pred)
        
        if verbose:
            print(f"\nDecision Tree Results:")
            print(f"  Fidelity (Test): {test_fidelity:.3f}")
            print(f"  Test Balanced: {tree_test_bal:.3f}")
            print(f"  Tree Depth: {tree.get_depth()}, Leaves: {tree.get_n_leaves()}")
        
        feature_importance = tree.feature_importances_
        top_features = np.argsort(feature_importance)[-10:][::-1]
        feature_names = (['A', 'B', 'C', 'D', 'E'] + 
                        [f'Sturm{i}' for i in range(8)] + [f'Desc{i}' for i in range(6)] +
                        [f'Newton{i}' for i in range(10)] + [f'Crit{i}' for i in range(10)] +
                        [f'Hybrid{i}' for i in range(16)] + [f'Decomp{i}' for i in range(8)])
        
        if verbose:
            print("\nTop 10 Features by Importance:")
            for idx in top_features:
                if feature_importance[idx] > 0.001:
                    print(f"  {feature_names[idx]:12s}: {feature_importance[idx]:.3f}")
            crit8_importance = feature_importance[37]
            print(f"\n*** Crit8 (Feature 37) importance: {crit8_importance:.3f} ***")
            if len(top_features) > 1:
                ratio = feature_importance[top_features[0]] / feature_importance[top_features[1]]
                print(f"Top feature is {ratio:.1f}× more important than 2nd")
        
        return {
            'nn_balanced': nn_bal, 'tree_fidelity': test_fidelity,
            'tree_balanced': tree_test_bal, 'crit8_importance': feature_importance[37]
        }
    
    def run_statistical_validation(self, n_trials=20, n_samples=5000):
        """20-trial statistical validation."""
        print("\n" + "="*80)
        print(f"Statistical validation: {n_trials} trails")
        print("="*80)
        
        nn_accs, tree_fids, tree_accs = [], [], []
        
        for trial in range(n_trials):
            print(f"\nTrial {trial+1}/{n_trials}", end=" ")
            extractor = QuinticRuleExtractor(random_state=trial)
            X, y = extractor.generate_quintic_data(n_samples)
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
            
            scaler = StandardScaler()
            X_tr_sc = scaler.fit_transform(X_tr)
            X_te_sc = scaler.transform(X_te)
            
            nn = MLPClassifier(hidden_layer_sizes=(200,100,50), max_iter=500, random_state=42)
            nn.fit(X_tr_sc, y_tr)
            nn_pred = nn.predict(X_te_sc)
            nn_acc = balanced_accuracy_score(y_te, nn_pred)
            nn_accs.append(nn_acc)
            
            tree = DecisionTreeClassifier(max_depth=8, random_state=42)
            tree.fit(X_tr, nn.predict(X_tr_sc))
            tree_pred = tree.predict(X_te)
            fid = accuracy_score(nn_pred, tree_pred)
            tree_acc = balanced_accuracy_score(y_te, tree_pred)
            tree_fids.append(fid)
            tree_accs.append(tree_acc)
            
            print(f"NN={nn_acc:.3f} Fid={fid:.3f} Tree={tree_acc:.3f}")
        
        nn_mean, nn_std = np.mean(nn_accs), np.std(nn_accs)
        fid_mean, fid_std = np.mean(tree_fids), np.std(tree_fids)
        tree_mean, tree_std = np.mean(tree_accs), np.std(tree_accs)
        
        print("\n" + "="*80)
        print("Statistical summary")
        print("="*80)
        print(f"\nNeural Network:")
        print(f"  Mean Test Balanced Accuracy: {nn_mean:.3f} ± {nn_std:.3f}")
        print(f"  Range: [{np.min(nn_accs):.3f}, {np.max(nn_accs):.3f}]")
        print(f"\nDecision Tree (Distillation):")
        print(f"  Mean Test Fidelity: {fid_mean:.3f} ± {fid_std:.3f}")
        print(f"  Mean Test Balanced Accuracy: {tree_mean:.3f} ± {tree_std:.3f}")
        print(f"  Fidelity Range: [{np.min(tree_fids):.3f}, {np.max(tree_fids):.3f}]")
        print(f"\nBest Trial: #{np.argmax(nn_accs)+1}")
        print(f"  NN Accuracy: {np.max(nn_accs):.3f}")
        print("="*80)
        
        return {
            'nn': (nn_mean, nn_std, nn_accs),
            'fidelity': (fid_mean, fid_std, tree_fids),
            'tree': (tree_mean, tree_std, tree_accs)
        }
    
    def run_complete_analysis(self, test_crit8=True, multi_trial_crit8=True,
                             test_crit8_dt=True, detailed=True, multi_trial=True, 
                             n_trials=20, n_samples=5000):
        """Run complete analysis with all tests."""
        print("\n" + "#"*80)
        print("# Quintic rule extraction")
        print("#"*80)
        
        results = {}
        if test_crit8:
            results['crit8_single'] = self.test_crit8_alone(n_samples=n_samples, verbose=True)
        if multi_trial_crit8:
            results['crit8_multi'] = self.test_crit8_alone_multi_trial(
                n_samples=n_samples, n_trials=n_trials, verbose=True
            )
        if test_crit8_dt:
            results['crit8_dt'] = self.test_crit8_decision_tree(
                n_samples=n_samples, n_trials=n_trials, verbose=True
            )
        if detailed:
            results['detailed'] = self.train_and_distill(n_samples=n_samples, verbose=True)
        if multi_trial:
            results['statistical'] = self.run_statistical_validation(
                n_trials=n_trials, n_samples=n_samples
            )
        return results

# Run it
if __name__ == "__main__":
    print("Starting Quintic Rule Extraction Analysis")
    
    extractor = QuinticRuleExtractor(random_state=42)
    results = extractor.run_complete_analysis(
        test_crit8=True,
        multi_trial_crit8=True,
        test_crit8_dt=True,
        detailed=True,
        multi_trial=True,
        n_trials=20,
        n_samples=5000
    )
    
    print("\nResults Summary:")
    print("-" * 40)
    
    if 'crit8_single' in results:
        print(f"Crit8 (seed=42): {results['crit8_single']['baseline']:.1%} → {results['crit8_single']['with_crit8']:.1%}")
    
    if 'crit8_multi' in results:
        print(f"Crit8 (20 trials): {results['crit8_multi']['baseline_mean']:.1%}±{results['crit8_multi']['baseline_std']:.1%} → {results['crit8_multi']['crit8_mean']:.1%}±{results['crit8_multi']['crit8_std']:.1%}")
        print(f"  Average improvement: +{results['crit8_multi']['improvement']:.1%}")
    
    if 'crit8_dt' in results:
        print(f"\nCrit8 Decision Trees (20 trials): {results['crit8_dt']['dt_baseline_mean']:.1%}±{results['crit8_dt']['dt_baseline_std']:.1%} → {results['crit8_dt']['dt_crit8_mean']:.1%}±{results['crit8_dt']['dt_crit8_std']:.1%}")
        print(f"  Average improvement: +{results['crit8_dt']['dt_improvement']:.1%}")
    
    if 'statistical' in results:
        nn_mean, nn_std, _ = results['statistical']['nn']
        fid_mean, fid_std, _ = results['statistical']['fidelity']
        tree_mean, tree_std, _ = results['statistical']['tree']
        print(f"\nNN: {nn_mean:.1%}±{nn_std:.1%}  |  Fidelity: {fid_mean:.1%}±{fid_std:.1%}  |  Tree: {tree_mean:.1%}±{tree_std:.1%}")